In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

np.random.seed(42)
from mmm_lab.data_generation.baseline import generate_baseline_geo_data, print_summary
from mmm_lab.data_generation.marketing import add_marketing_effects


In [ ]:
df = generate_baseline_geo_data(n_geos=40, n_weeks=104)
print_summary(df)


In [ ]:
proxies = ['demand_very_good', 'demand_good', 'demand_poor']
corrs = {p: df.groupby('geo').apply(
    lambda x: x['baseline_bookings'].corr(x[p]), include_groups=False
) for p in proxies}

fig, ax = plt.subplots(figsize=(8, 4))
for p, c in corrs.items():
    ax.hist(c, bins=15, alpha=0.6, label=f'{p} (mean={c.mean():.2f})')
ax.set_xlabel('Correlation with baseline_bookings')
ax.set_title('Demand Proxy Quality Distribution Across Geos')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
geo_df = df[df['geo'] == 0].set_index('date')

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
series = ['baseline_bookings', 'demand_very_good', 'demand_good', 'demand_poor']
colors = ['black', 'steelblue', 'orange', 'red']

for ax, col, color in zip(axes, series, colors):
    ax.plot(geo_df.index, geo_df[col], color=color, linewidth=1)
    ax.set_title(col)
    ax.set_ylabel('value')

plt.suptitle('Geo 0 — Demand Proxy Time Series', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, proxy, color in zip(axes, proxies, ['steelblue', 'orange', 'red']):
    r = df.groupby('geo').apply(lambda x: x['baseline_bookings'].corr(x[proxy]), include_groups=False).mean()
    ax.set_title(f'{proxy}\nr_within_geo={r:.3f}')
    ax.scatter(df['baseline_bookings'], df[proxy], alpha=0.05, s=5, color=color)
    ax.set_xlabel('baseline_bookings')

plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

df_full = add_marketing_effects(df.copy(), channels=['tv', 'paid_search'])

# Check residual autocorrelation on baseline for a few geos
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, geo in enumerate([0, 10, 20]):
    geo_data = df_full[df_full['geo'] == geo].sort_values('date')
    plot_acf(geo_data['baseline_bookings'], lags=20, ax=axes[i], zero=False)
    axes[i].set_title(f'Geo {geo} baseline ACF')
    axes[i].axhline(y=0, color='k', linewidth=0.5)

plt.suptitle('Baseline Autocorrelation (should be ~0 at all lags after AR(1) removal)')
plt.tight_layout()
plt.show()


In [ ]:
geo_stats = df_full.groupby('geo').agg(
    population=('population', 'first'),
    spend_tv=('spend_tv', 'sum'),
    effect_tv=('effect_tv', 'sum'),
    spend_ps=('spend_paid_search', 'sum'),
    effect_ps=('effect_paid_search', 'sum'),
)
for col in ['spend_tv', 'effect_tv', 'spend_ps', 'effect_ps']:
    geo_stats[col+'_pc'] = geo_stats[col] / geo_stats['population']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, ch in zip(axes, ['tv', 'ps']):
    x, y = geo_stats[f'spend_{ch}_pc'], geo_stats[f'effect_{ch}_pc']
    ax.scatter(x, y)
    r = x.corr(y)
    ax.set(title=f'{ch.upper()} per-capita spend vs effect  (r={r:.3f})',
           xlabel='spend/capita', ylabel='effect/capita')

plt.tight_layout()
plt.show()

print(f"Media share: {(df_full['effect_tv']+df_full['effect_paid_search']).sum()/df_full['total_bookings'].sum():.1%}")
true_tv = df_full.attrs['channel_params']['tv']['overall_roas']
true_ps = df_full.attrs['channel_params']['paid_search']['overall_roas']
print(f"TV ROAS:  {df_full['effect_tv'].sum()/df_full['spend_tv'].sum():.3f}  (true: {true_tv:.3f})")
print(f"PS ROAS:  {df_full['effect_paid_search'].sum()/df_full['spend_paid_search'].sum():.3f}  (true: {true_ps:.3f})")